# SQL Lab: MiniNaukri (companion to Part IX, Chapters 45-48)

Every query printed in the book's SQL chapters, runnable in Google Colab with **no installation**: SQLite ships with Python.

* Run all cells once (Runtime -> Run all). Each query runs on a **fresh copy** of the database, so the INSERT/UPDATE/DELETE examples
  never affect later queries.
* Before running a query, predict its output (number of rows, the first row). Then run it and compare with the book.
* The exercises at the end have solutions in the last section: try them first.

The database: `companies`, `candidates`, `jobs`, `applications`, `logins`, `impressions`, `employees` (see Chapter 45 for the diagram).

In [1]:
import sqlite3, re
import pandas as pd
pd.set_option('display.width', 160)

SCHEMA = r'''-- MiniNaukri: a tiny job-portal database used by every query in Part IX.
-- Runs as-is in SQLite (Python's sqlite3, Colab), PostgreSQL and MySQL 8.

DROP TABLE IF EXISTS impressions;
DROP TABLE IF EXISTS logins;
DROP TABLE IF EXISTS applications;
DROP TABLE IF EXISTS jobs;
DROP TABLE IF EXISTS candidates;
DROP TABLE IF EXISTS companies;
DROP TABLE IF EXISTS employees;

CREATE TABLE companies (
  company_id  INTEGER PRIMARY KEY,
  name        VARCHAR(30) NOT NULL,
  city        VARCHAR(20),
  industry    VARCHAR(20),
  size        VARCHAR(10)
);
INSERT INTO companies VALUES
 (1, 'DataNest',  'Bengaluru', 'IT',          'large'),
 (2, 'PayWave',   'Mumbai',    'Fintech',     'mid'),
 (3, 'MediCore',  'Pune',      'Healthcare',  'small'),
 (4, 'ShopKart',  'Bengaluru', 'E-commerce',  'large'),
 (5, 'EduSpark',  'Noida',     'Edtech',      'mid'),
 (6, 'GreenGrid', 'Delhi',     'Energy',      'small');

CREATE TABLE candidates (
  cand_id      INTEGER PRIMARY KEY,
  name         VARCHAR(20) NOT NULL,
  city         VARCHAR(20),          -- NULL = not filled in
  exp_yrs      INTEGER,
  ctc_lpa      DECIMAL(5,1),         -- current salary, lakh per annum; NULL for freshers
  signup_date  DATE,
  referred_by  INTEGER REFERENCES candidates(cand_id)
);
INSERT INTO candidates VALUES
 (101, 'Aarav',   'Bengaluru',  2,  8.0, '2026-01-05', NULL),
 (102, 'Diya',    'Mumbai',     5, 18.0, '2026-01-07', 101),
 (103, 'Kabir',   'Delhi',      0, NULL, '2026-01-10', NULL),
 (104, 'Meera',   'Bengaluru',  7, 30.0, '2026-01-12', 102),
 (105, 'Rohan',   'Pune',       3, 12.0, '2026-02-01', 101),
 (106, 'Sana',    NULL,         1,  5.5, '2026-02-03', NULL),
 (107, 'Vikram',  'Noida',     10, 42.0, '2026-02-15', 104),
 (108, 'Ananya',  'Mumbai',     4, 15.0, '2026-02-20', NULL),
 (109, 'Ishaan',  'Bengaluru',  2,  9.5, '2026-03-02', 105),
 (110, 'Priya',   'Delhi',      6, 24.0, '2026-03-05', NULL);

CREATE TABLE jobs (
  job_id       INTEGER PRIMARY KEY,
  company_id   INTEGER NOT NULL REFERENCES companies(company_id),
  title        VARCHAR(30),
  city         VARCHAR(20),
  min_exp      INTEGER,
  max_ctc_lpa  DECIMAL(5,1),
  posted_date  DATE
);
INSERT INTO jobs VALUES
 (201, 1, 'Data Scientist', 'Bengaluru', 2, 20, '2026-01-15'),
 (202, 1, 'ML Engineer',    'Bengaluru', 4, 35, '2026-01-20'),
 (203, 2, 'Data Analyst',   'Mumbai',    1, 12, '2026-01-25'),
 (204, 2, 'Data Scientist', 'Mumbai',    3, 25, '2026-02-05'),
 (205, 3, 'Data Analyst',   'Pune',      0,  8, '2026-02-10'),
 (206, 4, 'Data Scientist', 'Bengaluru', 5, 40, '2026-02-12'),
 (207, 4, 'Data Analyst',   'Bengaluru', 1, 10, '2026-02-18'),
 (208, 5, 'ML Engineer',    'Noida',     6, 45, '2026-03-01'),
 (209, 5, 'Data Scientist', 'Noida',     2, 18, '2026-03-04'),
 (210, 3, 'ML Engineer',    'Hyderabad', 3, 22, '2026-03-10');

CREATE TABLE applications (
  app_id        INTEGER PRIMARY KEY,
  cand_id       INTEGER NOT NULL REFERENCES candidates(cand_id),
  job_id        INTEGER NOT NULL REFERENCES jobs(job_id),
  applied_date  DATE,
  status        VARCHAR(12)   -- applied / shortlisted / interview / offer / rejected
);
INSERT INTO applications VALUES
 ( 1, 101, 201, '2026-01-16', 'interview'),
 ( 2, 101, 207, '2026-02-19', 'offer'),
 ( 3, 101, 209, '2026-03-05', 'applied'),
 ( 4, 102, 203, '2026-01-26', 'rejected'),
 ( 5, 102, 204, '2026-02-06', 'offer'),
 ( 6, 102, 201, '2026-01-18', 'shortlisted'),
 ( 7, 103, 205, '2026-02-11', 'shortlisted'),
 ( 8, 103, 203, '2026-01-27', 'rejected'),
 ( 9, 103, 207, '2026-02-19', 'applied'),
 (10, 104, 202, '2026-01-21', 'offer'),
 (11, 104, 206, '2026-02-13', 'interview'),
 (12, 105, 204, '2026-02-07', 'interview'),
 (13, 105, 205, '2026-02-11', 'rejected'),
 (14, 105, 201, '2026-02-02', 'applied'),
 (15, 107, 208, '2026-03-02', 'offer'),
 (16, 107, 206, '2026-02-16', 'rejected'),
 (17, 108, 203, '2026-02-21', 'interview'),
 (18, 108, 204, '2026-02-22', 'shortlisted'),
 (19, 109, 201, '2026-03-03', 'applied'),
 (20, 109, 209, '2026-03-05', 'shortlisted'),
 (21, 109, 207, '2026-03-03', 'rejected'),
 (22, 101, 201, '2026-01-16', 'interview'),   -- accidental duplicate of app 1
 (23, 104, 208, '2026-03-02', 'applied');

CREATE TABLE logins (
  cand_id     INTEGER REFERENCES candidates(cand_id),
  login_date  DATE
);
INSERT INTO logins VALUES
 (101,'2026-03-01'),(101,'2026-03-02'),(101,'2026-03-03'),(101,'2026-03-05'),(101,'2026-03-06'),
 (102,'2026-03-01'),(102,'2026-03-03'),
 (104,'2026-03-02'),(104,'2026-03-03'),(104,'2026-03-04'),(104,'2026-03-05'),
 (105,'2026-03-01'),
 (107,'2026-03-04'),(107,'2026-03-05'),(107,'2026-03-06'),
 (109,'2026-03-03'),(109,'2026-03-04'),(109,'2026-03-04');   -- 109 logged in twice on 03-04

CREATE TABLE impressions (            -- a job card shown to a candidate in an A/B test
  imp_id      INTEGER PRIMARY KEY,
  cand_id     INTEGER,
  job_id      INTEGER,
  shown_date  DATE,
  variant     CHAR(1),                -- 'A' = old ranking, 'B' = new ranking
  clicked     INTEGER                 -- 1 = clicked, 0 = not
);
INSERT INTO impressions VALUES
 ( 1,101,201,'2026-03-01','A',1),( 2,101,202,'2026-03-01','A',0),( 3,102,204,'2026-03-01','B',1),
 ( 4,102,203,'2026-03-01','B',0),( 5,103,205,'2026-03-02','A',0),( 6,103,207,'2026-03-02','A',0),
 ( 7,104,206,'2026-03-02','B',1),( 8,104,208,'2026-03-02','B',1),( 9,105,204,'2026-03-02','A',1),
 (10,105,210,'2026-03-03','A',0),(11,107,208,'2026-03-03','B',1),(12,107,206,'2026-03-03','B',0),
 (13,108,203,'2026-03-03','A',0),(14,108,204,'2026-03-03','A',0),(15,109,209,'2026-03-03','B',1),
 (16,109,207,'2026-03-03','B',0);

CREATE TABLE employees (               -- InfoEdge-style internal team table for the classics
  emp_id      INTEGER PRIMARY KEY,
  name        VARCHAR(20),
  dept        VARCHAR(20),
  salary      INTEGER,                 -- lakh per annum; NULL = not yet entered
  manager_id  INTEGER REFERENCES employees(emp_id),
  hire_date   DATE
);
INSERT INTO employees VALUES
 ( 1,'Neha',  'Leadership',  90, NULL,'2018-04-01'),
 ( 2,'Arjun', 'DataScience', 60, 1,   '2019-06-15'),
 ( 3,'Kavya', 'DataScience', 45, 2,   '2021-01-10'),
 ( 4,'Manav', 'DataScience', 45, 2,   '2022-03-01'),
 ( 5,'Tara',  'DataScience', 30, 2,   '2024-07-01'),
 ( 6,'Rahul', 'Engineering', 70, 1,   '2019-02-20'),
 ( 7,'Sneha', 'Engineering', 50, 6,   '2020-08-05'),
 ( 8,'Yash',  'Engineering', 75, 6,   '2023-05-12'),
 ( 9,'Pooja', 'Sales',       40, 1,   '2020-11-30'),
 (10,'Dev',   'Sales',       25, 9,   '2025-01-15'),
 (11,'Zoya',  'Sales',     NULL, 9,   '2026-01-05');
'''

In [2]:
def fresh_db():
    """A new in-memory database with the MiniNaukri tables."""
    con = sqlite3.connect(":memory:")
    con.executescript(SCHEMA)
    return con

def run(sql, con=None):
    """Run one or more SQL statements; return the result of the last one as a DataFrame."""
    con = con or fresh_db()
    stmts = [s.strip() for s in re.split(r";\s*\n", sql.strip().rstrip(";") + ";\n") if s.strip()]
    for s in stmts[:-1]:
        con.execute(s)
    cur = con.execute(stmts[-1])
    if cur.description is None:
        return "statement executed"
    return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])

db = fresh_db()   # a database you can play with freely: run("SELECT ...", db)
for t in ["companies", "candidates", "jobs", "applications", "logins", "impressions", "employees"]:
    print(t, run(f"SELECT COUNT(*) AS n FROM {t}", db).n[0])

companies 6
candidates 10
jobs 10
applications 23
logins 18
impressions 16
employees 11


## Look at the main tables first

In [3]:
run("SELECT * FROM candidates")

,cand_id,name,city,exp_yrs,ctc_lpa,signup_date,referred_by
0,101,Aarav,Bengaluru,2,8.0,2026-01-05,NaN
1,102,Diya,Mumbai,5,18.0,2026-01-07,101.0
2,103,Kabir,Delhi,0,NaN,2026-01-10,NaN
3,104,Meera,Bengaluru,7,30.0,2026-01-12,102.0
4,105,Rohan,Pune,3,12.0,2026-02-01,101.0
5,106,Sana,NaN,1,5.5,2026-02-03,NaN
6,107,Vikram,Noida,10,42.0,2026-02-15,104.0
7,108,Ananya,Mumbai,4,15.0,2026-02-20,NaN
8,109,Ishaan,Bengaluru,2,9.5,2026-03-02,105.0
9,110,Priya,Delhi,6,24.0,2026-03-05,NaN


In [4]:
run("SELECT * FROM jobs")

,job_id,company_id,title,city,min_exp,max_ctc_lpa,posted_date
0,201,1,Data Scientist,Bengaluru,2,20,2026-01-15
1,202,1,ML Engineer,Bengaluru,4,35,2026-01-20
2,203,2,Data Analyst,Mumbai,1,12,2026-01-25
3,204,2,Data Scientist,Mumbai,3,25,2026-02-05
4,205,3,Data Analyst,Pune,0,8,2026-02-10
5,206,4,Data Scientist,Bengaluru,5,40,2026-02-12
6,207,4,Data Analyst,Bengaluru,1,10,2026-02-18
7,208,5,ML Engineer,Noida,6,45,2026-03-01
8,209,5,Data Scientist,Noida,2,18,2026-03-04
9,210,3,ML Engineer,Hyderabad,3,22,2026-03-10


In [5]:
run("SELECT * FROM applications")

,app_id,cand_id,job_id,applied_date,status
0,1,101,201,2026-01-16,interview
1,2,101,207,2026-02-19,offer
2,3,101,209,2026-03-05,applied
3,4,102,203,2026-01-26,rejected
4,5,102,204,2026-02-06,offer
5,6,102,201,2026-01-18,shortlisted
6,7,103,205,2026-02-11,shortlisted
7,8,103,203,2026-01-27,rejected
8,9,103,207,2026-02-19,applied
9,10,104,202,2026-01-21,offer


# Level 1 (Chapter 45): one table: SELECT, WHERE, NULL, ORDER BY, CASE, strings, dates

### `L1_select_all`: SELECT * : every column

In [6]:
run("""
SELECT * FROM companies;
""")

,company_id,name,city,industry,size
0,1,DataNest,Bengaluru,IT,large
1,2,PayWave,Mumbai,Fintech,mid
2,3,MediCore,Pune,Healthcare,small
3,4,ShopKart,Bengaluru,E-commerce,large
4,5,EduSpark,Noida,Edtech,mid
5,6,GreenGrid,Delhi,Energy,small


### `L1_columns`: Choosing columns

In [7]:
run("""
SELECT name, city, exp_yrs
FROM candidates;
""")

,name,city,exp_yrs
0,Aarav,Bengaluru,2
1,Diya,Mumbai,5
2,Kabir,Delhi,0
3,Meera,Bengaluru,7
4,Rohan,Pune,3
5,Sana,NaN,1
6,Vikram,Noida,10
7,Ananya,Mumbai,4
8,Ishaan,Bengaluru,2
9,Priya,Delhi,6


### `L1_where`: WHERE: filtering rows

In [8]:
run("""
SELECT name, city, exp_yrs, ctc_lpa
FROM candidates
WHERE city = 'Bengaluru';
""")

,name,city,exp_yrs,ctc_lpa
0,Aarav,Bengaluru,2,8.0
1,Meera,Bengaluru,7,30.0
2,Ishaan,Bengaluru,2,9.5


### `L1_and_or`: AND/OR precedence bug

In [9]:
run("""
SELECT name, city, exp_yrs
FROM candidates
WHERE city = 'Bengaluru' OR city = 'Mumbai' AND exp_yrs >= 5;
""")

,name,city,exp_yrs
0,Aarav,Bengaluru,2
1,Diya,Mumbai,5
2,Meera,Bengaluru,7
3,Ishaan,Bengaluru,2


### `L1_and_or_fixed`: AND/OR fixed with brackets

In [10]:
run("""
SELECT name, city, exp_yrs
FROM candidates
WHERE (city = 'Bengaluru' OR city = 'Mumbai') AND exp_yrs >= 5;
""")

,name,city,exp_yrs
0,Diya,Mumbai,5
1,Meera,Bengaluru,7


### `L1_in_between`: IN and BETWEEN

In [11]:
run("""
SELECT name, city, exp_yrs
FROM candidates
WHERE city IN ('Delhi', 'Pune', 'Noida')
  AND exp_yrs BETWEEN 3 AND 10;
""")

,name,city,exp_yrs
0,Rohan,Pune,3
1,Vikram,Noida,10
2,Priya,Delhi,6


### `L1_like`: LIKE patterns

In [12]:
run("""
SELECT job_id, title, city
FROM jobs
WHERE title LIKE 'Data%';
""")

,job_id,title,city
0,201,Data Scientist,Bengaluru
1,203,Data Analyst,Mumbai
2,204,Data Scientist,Mumbai
3,205,Data Analyst,Pune
4,206,Data Scientist,Bengaluru
5,207,Data Analyst,Bengaluru
6,209,Data Scientist,Noida


### `L1_null_wrong`: = NULL finds nothing

In [13]:
run("""
SELECT name, city FROM candidates WHERE city = NULL;
""")

,name,city


### `L1_null_right`: IS NULL

In [14]:
run("""
SELECT name, city FROM candidates WHERE city IS NULL;
""")

,name,city
0,Sana,None


### `L1_not_equal_null`: <> silently drops NULL

In [15]:
run("""
SELECT COUNT(*) AS not_bengaluru
FROM candidates
WHERE city <> 'Bengaluru';
""")

,not_bengaluru
0,6


### `L1_order_limit`: ORDER BY + LIMIT

In [16]:
run("""
SELECT name, ctc_lpa
FROM candidates
ORDER BY ctc_lpa DESC
LIMIT 3;
""")

,name,ctc_lpa
0,Vikram,42
1,Meera,30
2,Priya,24


### `L1_order_nulls`: Where NULL sorts

In [17]:
run("""
SELECT name, ctc_lpa
FROM candidates
ORDER BY ctc_lpa ASC
LIMIT 3;
""")

,name,ctc_lpa
0,Kabir,NaN
1,Sana,5.5
2,Aarav,8.0


### `L1_offset`: OFFSET (paging)

In [18]:
run("""
SELECT name, ctc_lpa
FROM candidates
ORDER BY ctc_lpa DESC
LIMIT 3 OFFSET 3;
""")

,name,ctc_lpa
0,Diya,18
1,Ananya,15
2,Rohan,12


### `L1_distinct`: DISTINCT

In [19]:
run("""
SELECT DISTINCT title FROM jobs;
""")

,title
0,Data Scientist
1,ML Engineer
2,Data Analyst


### `L1_distinct_pair`: DISTINCT on two columns

In [20]:
run("""
SELECT DISTINCT title, city FROM jobs ORDER BY title, city;
""")

,title,city
0,Data Analyst,Bengaluru
1,Data Analyst,Mumbai
2,Data Analyst,Pune
3,Data Scientist,Bengaluru
4,Data Scientist,Mumbai
5,Data Scientist,Noida
6,ML Engineer,Bengaluru
7,ML Engineer,Hyderabad
8,ML Engineer,Noida


### `L1_computed`: Computed columns and aliases

In [21]:
run("""
SELECT name,
       ctc_lpa,
       ctc_lpa * 100000 / 12 AS monthly_ctc_rs,
       ctc_lpa * 1.30       AS expected_ctc_lpa
FROM candidates
WHERE ctc_lpa IS NOT NULL
ORDER BY expected_ctc_lpa DESC
LIMIT 4;
""")

,name,ctc_lpa,monthly_ctc_rs,expected_ctc_lpa
0,Vikram,42,350000,54.6
1,Meera,30,250000,39.0
2,Priya,24,200000,31.2
3,Diya,18,150000,23.4


### `L1_case`: CASE WHEN binning

In [22]:
run("""
SELECT name, exp_yrs,
       CASE
         WHEN exp_yrs = 0  THEN 'Fresher'
         WHEN exp_yrs <= 3 THEN 'Junior'
         WHEN exp_yrs <= 6 THEN 'Mid'
         ELSE 'Senior'
       END AS band
FROM candidates
ORDER BY exp_yrs;
""")

,name,exp_yrs,band
0,Kabir,0,Fresher
1,Sana,1,Junior
2,Aarav,2,Junior
3,Ishaan,2,Junior
4,Rohan,3,Junior
5,Ananya,4,Mid
6,Diya,5,Mid
7,Priya,6,Mid
8,Meera,7,Senior
9,Vikram,10,Senior


### `L1_coalesce`: COALESCE (fillna)

In [23]:
run("""
SELECT name,
       COALESCE(city, 'Unknown')  AS city,
       COALESCE(ctc_lpa, 0)       AS ctc_lpa_filled
FROM candidates
WHERE city IS NULL OR ctc_lpa IS NULL;
""")

,name,city,ctc_lpa_filled
0,Kabir,Delhi,0.0
1,Sana,Unknown,5.5


### `L1_strings`: String functions

In [24]:
run("""
SELECT name,
       UPPER(name)            AS upper_name,
       LENGTH(name)           AS n_chars,
       SUBSTR(name, 1, 3)     AS first3,
       name || ' (' || COALESCE(city, '?') || ')' AS label
FROM candidates
LIMIT 4;
""")

,name,upper_name,n_chars,first3,label
0,Aarav,AARAV,5,Aar,Aarav (Bengaluru)
1,Diya,DIYA,4,Diy,Diya (Mumbai)
2,Kabir,KABIR,5,Kab,Kabir (Delhi)
3,Meera,MEERA,5,Mee,Meera (Bengaluru)


### `L1_dates`: Date functions

In [25]:
run("""
SELECT job_id, posted_date,
       strftime('%Y-%m', posted_date)                  AS month,
       CAST(julianday('2026-03-15') - julianday(posted_date) AS INTEGER) AS days_open,
       date(posted_date, '+30 days')                   AS expires_on
FROM jobs
ORDER BY posted_date
LIMIT 4;
""")

,job_id,posted_date,month,days_open,expires_on
0,201,2026-01-15,2026-01,59,2026-02-14
1,202,2026-01-20,2026-01,54,2026-02-19
2,203,2026-01-25,2026-01,49,2026-02-24
3,204,2026-02-05,2026-02,38,2026-03-07


### `L1_intdiv`: Integer division trap

In [26]:
run("""
SELECT 7 / 2      AS int_div,
       7 * 1.0 / 2 AS real_div,
       ROUND(2.0 / 3, 3) AS rounded;
""")

,int_div,real_div,rounded
0,3,3.5,0.667


# Level 2 (Chapter 46): aggregation, GROUP BY, HAVING, joins, set operations

### `L2_agg_basic`: Aggregates and NULL

In [27]:
run("""
SELECT COUNT(*)        AS n_rows,
       COUNT(salary)   AS n_salary,
       SUM(salary)     AS total,
       AVG(salary)     AS avg_ignores_null,
       SUM(salary) * 1.0 / COUNT(*) AS avg_null_as_zero,
       MIN(salary)     AS min_sal,
       MAX(salary)     AS max_sal
FROM employees;
""")

,n_rows,n_salary,total,avg_ignores_null,avg_null_as_zero,min_sal,max_sal
0,11,10,530,53.0,48.181818,25,90


### `L2_count_distinct`: COUNT DISTINCT

In [28]:
run("""
SELECT COUNT(*)                AS applications,
       COUNT(DISTINCT cand_id) AS candidates_who_applied,
       COUNT(DISTINCT job_id)  AS jobs_with_applications
FROM applications;
""")

,applications,candidates_who_applied,jobs_with_applications
0,23,8,9


### `L2_groupby`: GROUP BY

In [29]:
run("""
SELECT title,
       COUNT(*)          AS n_jobs,
       AVG(max_ctc_lpa)  AS avg_max_ctc,
       MAX(max_ctc_lpa)  AS best_max_ctc
FROM jobs
GROUP BY title
ORDER BY avg_max_ctc DESC;
""")

,title,n_jobs,avg_max_ctc,best_max_ctc
0,ML Engineer,3,34.00,45
1,Data Scientist,4,25.75,40
2,Data Analyst,3,10.00,12


### `L2_groupby_two`: GROUP BY two columns

In [30]:
run("""
SELECT city, title, COUNT(*) AS n_jobs
FROM jobs
GROUP BY city, title
ORDER BY city, title;
""")

,city,title,n_jobs
0,Bengaluru,Data Analyst,1
1,Bengaluru,Data Scientist,2
2,Bengaluru,ML Engineer,1
3,Hyderabad,ML Engineer,1
4,Mumbai,Data Analyst,1
5,Mumbai,Data Scientist,1
6,Noida,Data Scientist,1
7,Noida,ML Engineer,1
8,Pune,Data Analyst,1


### `L2_having`: HAVING

In [31]:
run("""
SELECT cand_id, COUNT(*) AS n_apps
FROM applications
GROUP BY cand_id
HAVING COUNT(*) >= 3
ORDER BY n_apps DESC, cand_id;
""")

,cand_id,n_apps
0,101,4
1,102,3
2,103,3
3,104,3
4,105,3
5,109,3


### `L2_where_vs_having`: WHERE and HAVING together

In [32]:
run("""
SELECT cand_id, COUNT(*) AS n_rejections
FROM applications
WHERE status = 'rejected'
GROUP BY cand_id
HAVING COUNT(*) >= 1
ORDER BY cand_id;
""")

,cand_id,n_rejections
0,102,1
1,103,1
2,105,1
3,107,1
4,109,1


### `L2_conditional_agg`: Conditional aggregation (rates)

In [33]:
run("""
SELECT job_id,
       COUNT(*)                                            AS applicants,
       SUM(CASE WHEN status = 'offer'    THEN 1 ELSE 0 END) AS offers,
       SUM(CASE WHEN status = 'rejected' THEN 1 ELSE 0 END) AS rejections,
       ROUND(AVG(CASE WHEN status = 'offer' THEN 1.0 ELSE 0 END), 2) AS offer_rate
FROM applications
GROUP BY job_id
ORDER BY job_id;
""")

,job_id,applicants,offers,rejections,offer_rate
0,201,5,0,0,0.00
1,202,1,1,0,1.00
2,203,3,0,2,0.00
3,204,3,1,0,0.33
4,205,2,0,1,0.00
5,206,2,0,1,0.00
6,207,3,1,1,0.33
7,208,2,1,0,0.50
8,209,2,0,0,0.00


### `L2_status_counts`: Status counts

In [34]:
run("""
SELECT status, COUNT(*) AS n
FROM applications
GROUP BY status
ORDER BY n DESC;
""")

,status,n
0,rejected,5
1,interview,5
2,applied,5
3,shortlisted,4
4,offer,4


### `L2_inner_join`: INNER JOIN

In [35]:
run("""
SELECT j.job_id, j.title, c.name AS company, c.industry
FROM jobs AS j
JOIN companies AS c ON c.company_id = j.company_id
ORDER BY j.job_id
LIMIT 5;
""")

,job_id,title,company,industry
0,201,Data Scientist,DataNest,IT
1,202,ML Engineer,DataNest,IT
2,203,Data Analyst,PayWave,Fintech
3,204,Data Scientist,PayWave,Fintech
4,205,Data Analyst,MediCore,Healthcare


### `L2_three_join`: Chaining joins

In [36]:
run("""
SELECT a.app_id, ca.name AS candidate, j.title, co.name AS company, a.status
FROM applications a
JOIN candidates ca ON ca.cand_id    = a.cand_id
JOIN jobs       j  ON j.job_id      = a.job_id
JOIN companies  co ON co.company_id = j.company_id
WHERE a.status = 'offer'
ORDER BY a.app_id;
""")

,app_id,candidate,title,company,status
0,2,Aarav,Data Analyst,ShopKart,offer
1,5,Diya,Data Scientist,PayWave,offer
2,10,Meera,ML Engineer,DataNest,offer
3,15,Vikram,ML Engineer,EduSpark,offer


### `L2_left_join`: LEFT JOIN with zero counts

In [37]:
run("""
SELECT c.name AS company, COUNT(j.job_id) AS n_jobs
FROM companies c
LEFT JOIN jobs j ON j.company_id = c.company_id
GROUP BY c.company_id, c.name
ORDER BY n_jobs DESC, company;
""")

,company,n_jobs
0,DataNest,2
1,EduSpark,2
2,MediCore,2
3,PayWave,2
4,ShopKart,2
5,GreenGrid,0


### `L2_left_count_star`: COUNT(*) trap after LEFT JOIN

In [38]:
run("""
SELECT c.name AS company, COUNT(*) AS wrong_n_jobs
FROM companies c
LEFT JOIN jobs j ON j.company_id = c.company_id
GROUP BY c.company_id, c.name
ORDER BY wrong_n_jobs DESC, company;
""")

,company,wrong_n_jobs
0,DataNest,2
1,EduSpark,2
2,MediCore,2
3,PayWave,2
4,ShopKart,2
5,GreenGrid,1


### `L2_anti_join`: Anti-join

In [39]:
run("""
SELECT ca.cand_id, ca.name
FROM candidates ca
LEFT JOIN applications a ON a.cand_id = ca.cand_id
WHERE a.app_id IS NULL;
""")

,cand_id,name
0,106,Sana
1,110,Priya


### `L2_left_where_trap`: LEFT JOIN + WHERE trap

In [40]:
run("""
SELECT ca.name, a.job_id, a.status
FROM candidates ca
LEFT JOIN applications a ON a.cand_id = ca.cand_id
WHERE a.status = 'offer';
""")

,name,job_id,status
0,Aarav,207,offer
1,Diya,204,offer
2,Meera,202,offer
3,Vikram,208,offer


### `L2_left_on_filter`: Filter inside ON

In [41]:
run("""
SELECT ca.name, a.job_id AS offer_job
FROM candidates ca
LEFT JOIN applications a
       ON a.cand_id = ca.cand_id AND a.status = 'offer'
ORDER BY ca.cand_id;
""")

,name,offer_job
0,Aarav,207.0
1,Diya,204.0
2,Kabir,NaN
3,Meera,202.0
4,Rohan,NaN
5,Sana,NaN
6,Vikram,208.0
7,Ananya,NaN
8,Ishaan,NaN
9,Priya,NaN


### `L2_full_join`: FULL OUTER JOIN

In [42]:
run("""
WITH job_cities AS (
  SELECT city, COUNT(*) AS n_jobs FROM jobs GROUP BY city
),
cand_cities AS (
  SELECT city, COUNT(*) AS n_cands FROM candidates WHERE city IS NOT NULL GROUP BY city
)
SELECT COALESCE(j.city, c.city) AS city, j.n_jobs, c.n_cands
FROM job_cities j
FULL OUTER JOIN cand_cities c ON c.city = j.city
ORDER BY city;
""")

,city,n_jobs,n_cands
0,Bengaluru,4.0,3.0
1,Delhi,NaN,2.0
2,Hyderabad,1.0,NaN
3,Mumbai,2.0,2.0
4,Noida,2.0,1.0
5,Pune,1.0,1.0


### `L2_self_join`: SELF JOIN (manager)

In [43]:
run("""
SELECT e.name AS employee, e.salary, m.name AS manager, m.salary AS manager_salary
FROM employees e
LEFT JOIN employees m ON m.emp_id = e.manager_id
ORDER BY e.emp_id;
""")

,employee,salary,manager,manager_salary
0,Neha,90.0,NaN,NaN
1,Arjun,60.0,Neha,90.0
2,Kavya,45.0,Arjun,60.0
3,Manav,45.0,Arjun,60.0
4,Tara,30.0,Arjun,60.0
5,Rahul,70.0,Neha,90.0
6,Sneha,50.0,Rahul,70.0
7,Yash,75.0,Rahul,70.0
8,Pooja,40.0,Neha,90.0
9,Dev,25.0,Pooja,40.0


### `L2_referral_self`: SELF JOIN (referrals)

In [44]:
run("""
SELECT c.name AS candidate, r.name AS referred_by
FROM candidates c
JOIN candidates r ON r.cand_id = c.referred_by
ORDER BY c.cand_id;
""")

,candidate,referred_by
0,Diya,Aarav
1,Meera,Diya
2,Rohan,Aarav
3,Vikram,Meera
4,Ishaan,Rohan


### `L2_cross_join`: CROSS JOIN grid

In [45]:
run("""
SELECT v.variant, d.shown_date
FROM (SELECT DISTINCT variant FROM impressions) v
CROSS JOIN (SELECT DISTINCT shown_date FROM impressions) d
ORDER BY v.variant, d.shown_date;
""")

,variant,shown_date
0,A,2026-03-01
1,A,2026-03-02
2,A,2026-03-03
3,B,2026-03-01
4,B,2026-03-02
5,B,2026-03-03


### `L2_fanout`: Fan-out after join

In [46]:
run("""
SELECT SUM(j.max_ctc_lpa) AS sum_after_join, COUNT(*) AS rows_after_join
FROM jobs j
JOIN applications a ON a.job_id = j.job_id;
""")

,sum_after_join,rows_after_join
0,498,23


### `L2_fanout_truth`: The true sum

In [47]:
run("""
SELECT SUM(max_ctc_lpa) AS true_sum, COUNT(*) AS n_jobs FROM jobs;
""")

,true_sum,n_jobs
0,235,10


### `L2_join_eligibility`: Non-equi join

In [48]:
run("""
SELECT ca.name, j.job_id, j.title, j.min_exp
FROM candidates ca
JOIN jobs j
  ON ca.exp_yrs >= j.min_exp
 AND ca.city = j.city
ORDER BY ca.name, j.job_id;
""")

,name,job_id,title,min_exp
0,Aarav,201,Data Scientist,2
1,Aarav,207,Data Analyst,1
2,Ananya,203,Data Analyst,1
3,Ananya,204,Data Scientist,3
4,Diya,203,Data Analyst,1
5,Diya,204,Data Scientist,3
6,Ishaan,201,Data Scientist,2
7,Ishaan,207,Data Analyst,1
8,Meera,201,Data Scientist,2
9,Meera,202,ML Engineer,4


### `L2_union`: UNION

In [49]:
run("""
SELECT city FROM companies
UNION
SELECT city FROM candidates WHERE city IS NOT NULL
ORDER BY city;
""")

,city
0,Bengaluru
1,Delhi
2,Mumbai
3,Noida
4,Pune


### `L2_union_all`: UNION ALL

In [50]:
run("""
SELECT 'company' AS source, city FROM companies WHERE city = 'Bengaluru'
UNION ALL
SELECT 'candidate', city FROM candidates WHERE city = 'Bengaluru';
""")

,source,city
0,company,Bengaluru
1,company,Bengaluru
2,candidate,Bengaluru
3,candidate,Bengaluru
4,candidate,Bengaluru


### `L2_except`: EXCEPT

In [51]:
run("""
SELECT city FROM candidates
EXCEPT
SELECT city FROM jobs;
""")

,city
0,NaN
1,Delhi


# Level 3 (Chapter 47): subqueries, CTEs, window functions, changing data

### `L3_scalar_sub`: Scalar subquery

In [52]:
run("""
SELECT name, ctc_lpa
FROM candidates
WHERE ctc_lpa > (SELECT AVG(ctc_lpa) FROM candidates)
ORDER BY ctc_lpa DESC;
""")

,name,ctc_lpa
0,Vikram,42
1,Meera,30
2,Priya,24


### `L3_in_sub`: IN subquery

In [53]:
run("""
SELECT name
FROM candidates
WHERE cand_id IN (SELECT cand_id FROM applications WHERE status = 'offer')
ORDER BY name;
""")

,name
0,Aarav
1,Diya
2,Meera
3,Vikram


### `L3_not_in_null`: NOT IN with NULL returns nothing

In [54]:
run("""
SELECT name FROM employees
WHERE emp_id NOT IN (SELECT manager_id FROM employees);
""")

,name


### `L3_not_exists`: NOT EXISTS

In [55]:
run("""
SELECT e.name FROM employees e
WHERE NOT EXISTS (SELECT 1 FROM employees x WHERE x.manager_id = e.emp_id)
ORDER BY e.emp_id;
""")

,name
0,Kavya
1,Manav
2,Tara
3,Sneha
4,Yash
5,Dev
6,Zoya


### `L3_correlated`: Correlated subquery

In [56]:
run("""
SELECT e.name, e.dept, e.salary
FROM employees e
WHERE e.salary > (SELECT AVG(x.salary) FROM employees x WHERE x.dept = e.dept)
ORDER BY e.dept, e.salary DESC;
""")

,name,dept,salary
0,Arjun,DataScience,60
1,Yash,Engineering,75
2,Rahul,Engineering,70
3,Pooja,Sales,40


### `L3_derived_table`: Derived table

In [57]:
run("""
SELECT band, COUNT(*) AS n, AVG(ctc_lpa) AS avg_ctc
FROM (
  SELECT ctc_lpa,
         CASE WHEN exp_yrs <= 3 THEN '0-3' WHEN exp_yrs <= 6 THEN '4-6' ELSE '7+' END AS band
  FROM candidates
) t
GROUP BY band
ORDER BY band;
""")

,band,n,avg_ctc
0,0-3,5,8.75
1,4-6,3,19.00
2,7+,2,36.00


### `L3_cte`: CTEs

In [58]:
run("""
WITH apps_per_job AS (
  SELECT job_id, COUNT(*) AS n_apps
  FROM applications
  GROUP BY job_id
),
job_info AS (
  SELECT j.job_id, j.title, c.name AS company, COALESCE(p.n_apps, 0) AS n_apps
  FROM jobs j
  JOIN companies c ON c.company_id = j.company_id
  LEFT JOIN apps_per_job p ON p.job_id = j.job_id
)
SELECT * FROM job_info
WHERE n_apps >= (SELECT AVG(n_apps) FROM job_info)
ORDER BY n_apps DESC, job_id;
""")

,job_id,title,company,n_apps
0,201,Data Scientist,DataNest,5
1,203,Data Analyst,PayWave,3
2,204,Data Scientist,PayWave,3
3,207,Data Analyst,ShopKart,3


### `L3_recursive_hier`: Recursive CTE: org chart

In [59]:
run("""
WITH RECURSIVE chain(emp_id, name, manager_id, level, path) AS (
  SELECT emp_id, name, manager_id, 0, name
  FROM employees WHERE manager_id IS NULL
  UNION ALL
  SELECT e.emp_id, e.name, e.manager_id, c.level + 1, c.path || ' > ' || e.name
  FROM employees e
  JOIN chain c ON e.manager_id = c.emp_id
)
SELECT level, name, path FROM chain ORDER BY path;
""")

,level,name,path
0,0,Neha,Neha
1,1,Arjun,Neha > Arjun
2,2,Kavya,Neha > Arjun > Kavya
3,2,Manav,Neha > Arjun > Manav
4,2,Tara,Neha > Arjun > Tara
5,1,Pooja,Neha > Pooja
6,2,Dev,Neha > Pooja > Dev
7,2,Zoya,Neha > Pooja > Zoya
8,1,Rahul,Neha > Rahul
9,2,Sneha,Neha > Rahul > Sneha


### `L3_recursive_dates`: Recursive CTE: date spine and DAU

In [60]:
run("""
WITH RECURSIVE days(d) AS (
  SELECT '2026-03-01'
  UNION ALL
  SELECT date(d, '+1 day') FROM days WHERE d < '2026-03-07'
)
SELECT days.d AS day, COUNT(DISTINCT l.cand_id) AS dau
FROM days
LEFT JOIN logins l ON l.login_date = days.d
GROUP BY days.d
ORDER BY days.d;
""")

,day,dau
0,2026-03-01,3
1,2026-03-02,2
2,2026-03-03,4
3,2026-03-04,3
4,2026-03-05,3
5,2026-03-06,2
6,2026-03-07,0


### `L3_rownum_rank`: ROW_NUMBER vs RANK vs DENSE_RANK

In [61]:
run("""
SELECT name, dept, salary,
       ROW_NUMBER() OVER (ORDER BY salary DESC) AS row_num,
       RANK()       OVER (ORDER BY salary DESC) AS rnk,
       DENSE_RANK() OVER (ORDER BY salary DESC) AS dense_rnk
FROM employees
WHERE salary IS NOT NULL
ORDER BY salary DESC, name;
""")

,name,dept,salary,row_num,rnk,dense_rnk
0,Neha,Leadership,90,1,1,1
1,Yash,Engineering,75,2,2,2
2,Rahul,Engineering,70,3,3,3
3,Arjun,DataScience,60,4,4,4
4,Sneha,Engineering,50,5,5,5
5,Kavya,DataScience,45,6,6,6
6,Manav,DataScience,45,7,6,6
7,Pooja,Sales,40,8,8,7
8,Tara,DataScience,30,9,9,8
9,Dev,Sales,25,10,10,9


### `L3_partition`: PARTITION BY

In [62]:
run("""
SELECT name, dept, salary,
       DENSE_RANK() OVER (PARTITION BY dept ORDER BY salary DESC) AS rank_in_dept,
       AVG(salary)  OVER (PARTITION BY dept)                      AS dept_avg,
       salary - AVG(salary) OVER (PARTITION BY dept)              AS diff_from_avg
FROM employees
WHERE salary IS NOT NULL
ORDER BY dept, salary DESC, name;
""")

,name,dept,salary,rank_in_dept,dept_avg,diff_from_avg
0,Arjun,DataScience,60,1,45.0,15.0
1,Kavya,DataScience,45,2,45.0,0.0
2,Manav,DataScience,45,2,45.0,0.0
3,Tara,DataScience,30,3,45.0,-15.0
4,Yash,Engineering,75,1,65.0,10.0
5,Rahul,Engineering,70,2,65.0,5.0
6,Sneha,Engineering,50,3,65.0,-15.0
7,Neha,Leadership,90,1,90.0,0.0
8,Pooja,Sales,40,1,32.5,7.5
9,Dev,Sales,25,2,32.5,-7.5


### `L3_groupby_vs_window`: GROUP BY collapses rows

In [63]:
run("""
SELECT dept, AVG(salary) AS dept_avg
FROM employees
GROUP BY dept
ORDER BY dept;
""")

,dept,dept_avg
0,DataScience,45.0
1,Engineering,65.0
2,Leadership,90.0
3,Sales,32.5


### `L3_lag_lead`: LAG and LEAD

In [64]:
run("""
SELECT cand_id, applied_date,
       LAG(applied_date)  OVER (PARTITION BY cand_id ORDER BY applied_date, app_id) AS prev_app,
       LEAD(applied_date) OVER (PARTITION BY cand_id ORDER BY applied_date, app_id) AS next_app,
       CAST(julianday(applied_date)
            - julianday(LAG(applied_date) OVER (PARTITION BY cand_id ORDER BY applied_date, app_id))
            AS INTEGER) AS days_since_prev
FROM applications
WHERE cand_id IN (101, 102)
ORDER BY cand_id, applied_date, app_id;
""")

,cand_id,applied_date,prev_app,next_app,days_since_prev
0,101,2026-01-16,NaN,2026-01-16,NaN
1,101,2026-01-16,2026-01-16,2026-02-19,0.0
2,101,2026-02-19,2026-01-16,2026-03-05,34.0
3,101,2026-03-05,2026-02-19,NaN,14.0
4,102,2026-01-18,NaN,2026-01-26,NaN
5,102,2026-01-26,2026-01-18,2026-02-06,8.0
6,102,2026-02-06,2026-01-26,NaN,11.0


### `L3_running`: Running total and moving average

In [65]:
run("""
WITH daily AS (
  SELECT applied_date AS day, COUNT(*) AS n
  FROM applications
  WHERE applied_date >= '2026-02-01' AND applied_date < '2026-03-01'
  GROUP BY applied_date
)
SELECT day, n,
       SUM(n) OVER (ORDER BY day)                                        AS running_total,
       ROUND(AVG(n) OVER (ORDER BY day ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) AS moving_avg_3
FROM daily
ORDER BY day;
""")

,day,n,running_total,moving_avg_3
0,2026-02-02,1,1,1.00
1,2026-02-06,1,2,1.00
2,2026-02-07,1,3,1.00
3,2026-02-11,2,5,1.33
4,2026-02-13,1,6,1.33
5,2026-02-16,1,7,1.33
6,2026-02-19,2,9,1.33
7,2026-02-21,1,10,1.33
8,2026-02-22,1,11,1.33


### `L3_pct_total`: Percent of total

In [66]:
run("""
SELECT title, COUNT(*) AS n_jobs,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_jobs
FROM jobs
GROUP BY title
ORDER BY n_jobs DESC;
""")

,title,n_jobs,pct_of_jobs
0,Data Scientist,4,40.0
1,Data Analyst,3,30.0
2,ML Engineer,3,30.0


### `L3_ntile_first`: NTILE, FIRST_VALUE, PERCENT_RANK

In [67]:
run("""
SELECT name, ctc_lpa,
       NTILE(4) OVER (ORDER BY ctc_lpa)                       AS quartile,
       FIRST_VALUE(name) OVER (ORDER BY ctc_lpa DESC)         AS top_earner,
       ROUND(PERCENT_RANK() OVER (ORDER BY ctc_lpa), 3)       AS pct_rank
FROM candidates
WHERE ctc_lpa IS NOT NULL
ORDER BY ctc_lpa;
""")

,name,ctc_lpa,quartile,top_earner,pct_rank
0,Sana,5.5,1,Vikram,0.000
1,Aarav,8.0,1,Vikram,0.125
2,Ishaan,9.5,1,Vikram,0.250
3,Rohan,12.0,2,Vikram,0.375
4,Ananya,15.0,2,Vikram,0.500
5,Diya,18.0,3,Vikram,0.625
6,Priya,24.0,3,Vikram,0.750
7,Meera,30.0,4,Vikram,0.875
8,Vikram,42.0,4,Vikram,1.000


### `L3_frame_trap`: LAST_VALUE frame trap

In [68]:
run("""
SELECT name, salary,
       LAST_VALUE(name) OVER (ORDER BY salary)  AS last_default,
       LAST_VALUE(name) OVER (ORDER BY salary
                 ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS last_full
FROM employees
WHERE dept = 'DataScience'
ORDER BY salary;
""")

,name,salary,last_default,last_full
0,Tara,30,Tara,Arjun
1,Kavya,45,Manav,Arjun
2,Manav,45,Manav,Arjun
3,Arjun,60,Arjun,Arjun


### `L3_dml`: INSERT, UPDATE, DELETE

In [69]:
run("""
INSERT INTO applications (app_id, cand_id, job_id, applied_date, status)
VALUES (24, 110, 210, '2026-03-11', 'applied');
UPDATE applications SET status = 'shortlisted' WHERE app_id = 24;
DELETE FROM applications WHERE app_id = 22;
SELECT app_id, cand_id, job_id, status FROM applications WHERE app_id IN (1, 22, 24);
""")

,app_id,cand_id,job_id,status
0,1,101,201,interview
1,24,110,210,shortlisted


### `L3_ddl`: CREATE TABLE with constraints

In [70]:
run("""
CREATE TABLE skills (
  cand_id INTEGER NOT NULL REFERENCES candidates(cand_id),
  skill   VARCHAR(20) NOT NULL,
  level   INTEGER CHECK (level BETWEEN 1 AND 5),
  PRIMARY KEY (cand_id, skill)
);
INSERT INTO skills VALUES (101,'python',4),(101,'sql',3),(102,'sql',5);
SELECT * FROM skills;
""")

,cand_id,skill,level
0,101,python,4
1,101,sql,3
2,102,sql,5


# Level 4 (Chapter 48): interview patterns, ML feature tables, query plans

### `L4_nth_dense`: Nth highest: DENSE_RANK

In [71]:
run("""
SELECT DISTINCT salary
FROM (SELECT salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS dr
      FROM employees WHERE salary IS NOT NULL) t
WHERE dr = 3;
""")

,salary
0,70


### `L4_nth_limit`: Nth highest: LIMIT OFFSET

In [72]:
run("""
SELECT DISTINCT salary
FROM employees
WHERE salary IS NOT NULL
ORDER BY salary DESC
LIMIT 1 OFFSET 2;
""")

,salary
0,70


### `L4_nth_correlated`: Nth highest: correlated

In [73]:
run("""
SELECT DISTINCT e.salary
FROM employees e
WHERE 2 = (SELECT COUNT(DISTINCT x.salary) FROM employees x WHERE x.salary > e.salary);
""")

,salary
0,70


### `L4_second_max`: Second highest with MAX

In [74]:
run("""
SELECT MAX(salary) AS second_highest
FROM employees
WHERE salary < (SELECT MAX(salary) FROM employees);
""")

,second_highest
0,75


### `L4_topn_group`: Top N per group

In [75]:
run("""
WITH ranked AS (
  SELECT name, dept, salary,
         DENSE_RANK() OVER (PARTITION BY dept ORDER BY salary DESC) AS dr
  FROM employees
  WHERE salary IS NOT NULL
)
SELECT dept, name, salary
FROM ranked
WHERE dr <= 2
ORDER BY dept, salary DESC, name;
""")

,dept,name,salary
0,DataScience,Arjun,60
1,DataScience,Kavya,45
2,DataScience,Manav,45
3,Engineering,Yash,75
4,Engineering,Rahul,70
5,Leadership,Neha,90
6,Sales,Pooja,40
7,Sales,Dev,25


### `L4_dupes_find`: Find duplicates

In [76]:
run("""
SELECT cand_id, job_id, applied_date, COUNT(*) AS copies
FROM applications
GROUP BY cand_id, job_id, applied_date
HAVING COUNT(*) > 1;
""")

,cand_id,job_id,applied_date,copies
0,101,201,2026-01-16,2


### `L4_dupes_delete`: Delete duplicates

In [77]:
run("""
DELETE FROM applications
WHERE app_id NOT IN (
  SELECT MIN(app_id) FROM applications
  GROUP BY cand_id, job_id, applied_date
);
SELECT COUNT(*) AS rows_left FROM applications;
""")

,rows_left
0,22


### `L4_dupes_rownum`: Label duplicates with ROW_NUMBER

In [78]:
run("""
WITH r AS (
  SELECT app_id, cand_id, job_id,
         ROW_NUMBER() OVER (PARTITION BY cand_id, job_id, applied_date ORDER BY app_id) AS rn
  FROM applications
)
SELECT app_id, cand_id, job_id, rn FROM r WHERE cand_id = 101 ORDER BY app_id;
""")

,app_id,cand_id,job_id,rn
0,1,101,201,1
1,2,101,207,1
2,3,101,209,1
3,22,101,201,2


### `L4_more_than_manager`: Earns more than manager

In [79]:
run("""
SELECT e.name AS employee, e.salary, m.name AS manager, m.salary AS manager_salary
FROM employees e
JOIN employees m ON m.emp_id = e.manager_id
WHERE e.salary > m.salary;
""")

,employee,salary,manager,manager_salary
0,Yash,75,Rahul,70


### `L4_streaks`: Consecutive-day streaks

In [80]:
run("""
WITH d AS (SELECT DISTINCT cand_id, login_date FROM logins),
g AS (
  SELECT cand_id, login_date,
         julianday(login_date)
           - ROW_NUMBER() OVER (PARTITION BY cand_id ORDER BY login_date) AS grp
  FROM d
)
SELECT cand_id, MIN(login_date) AS streak_start, MAX(login_date) AS streak_end,
       COUNT(*) AS streak_len
FROM g
GROUP BY cand_id, grp
HAVING COUNT(*) >= 3
ORDER BY streak_len DESC, cand_id;
""")

,cand_id,streak_start,streak_end,streak_len
0,104,2026-03-02,2026-03-05,4
1,101,2026-03-01,2026-03-03,3
2,107,2026-03-04,2026-03-06,3


### `L4_streak_steps`: Gaps-and-islands, step by step

In [81]:
run("""
WITH d AS (SELECT DISTINCT cand_id, login_date FROM logins WHERE cand_id = 101)
SELECT login_date,
       ROW_NUMBER() OVER (ORDER BY login_date) AS rn,
       date(login_date, '-' || ROW_NUMBER() OVER (ORDER BY login_date) || ' days') AS grp_date
FROM d ORDER BY login_date;
""")

,login_date,rn,grp_date
0,2026-03-01,1,2026-02-28
1,2026-03-02,2,2026-02-28
2,2026-03-03,3,2026-02-28
3,2026-03-05,4,2026-03-01
4,2026-03-06,5,2026-03-01


### `L4_funnel`: Hiring funnel

In [82]:
run("""
WITH stage AS (
  SELECT app_id,
         CASE status WHEN 'applied' THEN 1 WHEN 'rejected' THEN 1
                     WHEN 'shortlisted' THEN 2 WHEN 'interview' THEN 3
                     WHEN 'offer' THEN 4 END AS reached
  FROM applications
  WHERE app_id <> 22
)
SELECT SUM(reached >= 1) AS applied,
       SUM(reached >= 2) AS shortlisted,
       SUM(reached >= 3) AS interviewed,
       SUM(reached >= 4) AS offered,
       ROUND(1.0 * SUM(reached >= 2) / SUM(reached >= 1), 3) AS apply_to_short,
       ROUND(1.0 * SUM(reached >= 4) / SUM(reached >= 3), 3) AS interview_to_offer
FROM stage;
""")

,applied,shortlisted,interviewed,offered,apply_to_short,interview_to_offer
0,22,12,8,4,0.545,0.5


### `L4_ab_ctr`: A/B test CTR

In [83]:
run("""
SELECT variant,
       COUNT(*)                    AS impressions,
       SUM(clicked)                AS clicks,
       ROUND(AVG(clicked * 1.0), 3) AS ctr
FROM impressions
GROUP BY variant;
""")

,variant,impressions,clicks,ctr
0,A,8,2,0.250
1,B,8,5,0.625


### `L4_ctr_daily`: CTR by day and variant

In [84]:
run("""
SELECT shown_date, variant, COUNT(*) AS imps, SUM(clicked) AS clicks,
       ROUND(1.0 * SUM(clicked) / COUNT(*), 2) AS ctr
FROM impressions
GROUP BY shown_date, variant
ORDER BY shown_date, variant;
""")

,shown_date,variant,imps,clicks,ctr
0,2026-03-01,A,2,1,0.50
1,2026-03-01,B,2,1,0.50
2,2026-03-02,A,3,1,0.33
3,2026-03-02,B,2,2,1.00
4,2026-03-03,A,3,0,0.00
5,2026-03-03,B,4,2,0.50


### `L4_cohort`: Cohort retention

In [85]:
run("""
WITH cohort AS (
  SELECT cand_id, strftime('%Y-%m', signup_date) AS signup_month FROM candidates
),
active AS (
  SELECT DISTINCT cand_id FROM logins
  WHERE login_date BETWEEN '2026-03-01' AND '2026-03-07'
)
SELECT c.signup_month,
       COUNT(*)                         AS signed_up,
       COUNT(a.cand_id)                 AS active_first_week_march,
       ROUND(1.0 * COUNT(a.cand_id) / COUNT(*), 2) AS retention
FROM cohort c
LEFT JOIN active a ON a.cand_id = c.cand_id
GROUP BY c.signup_month
ORDER BY c.signup_month;
""")

,signup_month,signed_up,active_first_week_march,retention
0,2026-01,4,3,0.75
1,2026-02,4,2,0.50
2,2026-03,2,1,0.50


### `L4_mom`: Month-over-month growth

In [86]:
run("""
WITH monthly AS (
  SELECT strftime('%Y-%m', applied_date) AS month, COUNT(*) AS n_apps
  FROM applications
  GROUP BY 1
)
SELECT month, n_apps,
       LAG(n_apps) OVER (ORDER BY month) AS prev_month,
       ROUND(100.0 * (n_apps - LAG(n_apps) OVER (ORDER BY month))
                   / LAG(n_apps) OVER (ORDER BY month), 1) AS mom_growth_pct
FROM monthly
ORDER BY month;
""")

,month,n_apps,prev_month,mom_growth_pct
0,2026-01,6,NaN,NaN
1,2026-02,11,6.0,83.3
2,2026-03,6,11.0,-45.5


### `L4_median`: Median

In [87]:
run("""
WITH s AS (
  SELECT ctc_lpa,
         ROW_NUMBER() OVER (ORDER BY ctc_lpa) AS rn,
         COUNT(*)     OVER ()                 AS n
  FROM candidates
  WHERE ctc_lpa IS NOT NULL
)
SELECT AVG(ctc_lpa) AS median_ctc
FROM s
WHERE rn IN ((n + 1) / 2, (n + 2) / 2);
""")

,median_ctc
0,15.0


### `L4_pivot`: Pivot

In [88]:
run("""
SELECT co.name AS company,
       SUM(CASE WHEN j.title = 'Data Scientist' THEN 1 ELSE 0 END) AS data_scientist,
       SUM(CASE WHEN j.title = 'Data Analyst'   THEN 1 ELSE 0 END) AS data_analyst,
       SUM(CASE WHEN j.title = 'ML Engineer'    THEN 1 ELSE 0 END) AS ml_engineer
FROM companies co
LEFT JOIN jobs j ON j.company_id = co.company_id
GROUP BY co.company_id, co.name
ORDER BY co.company_id;
""")

,company,data_scientist,data_analyst,ml_engineer
0,DataNest,1,0,1
1,PayWave,1,1,0
2,MediCore,0,1,1
3,ShopKart,1,1,0
4,EduSpark,1,0,1
5,GreenGrid,0,0,0


### `L4_first_app`: First event per entity

In [89]:
run("""
SELECT cand_id, job_id, applied_date
FROM (
  SELECT a.*, ROW_NUMBER() OVER (PARTITION BY cand_id ORDER BY applied_date, app_id) AS rn
  FROM applications a
) t
WHERE rn = 1
ORDER BY cand_id;
""")

,cand_id,job_id,applied_date
0,101,201,2026-01-16
1,102,201,2026-01-18
2,103,203,2026-01-27
3,104,202,2026-01-21
4,105,201,2026-02-02
5,107,206,2026-02-16
6,108,203,2026-02-21
7,109,201,2026-03-03


### `L4_time_to_apply`: Time between events

In [90]:
run("""
SELECT j.title,
       ROUND(AVG(julianday(a.applied_date) - julianday(j.posted_date)), 2) AS avg_days_to_apply,
       COUNT(*) AS n_apps
FROM applications a
JOIN jobs j ON j.job_id = a.job_id
WHERE a.app_id <> 22
GROUP BY j.title
ORDER BY avg_days_to_apply;
""")

,title,avg_days_to_apply,n_apps
0,ML Engineer,1.00,3
1,Data Analyst,5.88,8
2,Data Scientist,8.73,11


### `L4_division`: Applied to all (relational division)

In [91]:
run("""
SELECT a.cand_id
FROM applications a
JOIN jobs j ON j.job_id = a.job_id
WHERE j.title = 'Data Analyst'
GROUP BY a.cand_id
HAVING COUNT(DISTINCT a.job_id) = (SELECT COUNT(*) FROM jobs WHERE title = 'Data Analyst');
""")

,cand_id
0,103


### `L4_rolling_users`: Rolling 3-day active users

In [92]:
run("""
WITH d AS (SELECT DISTINCT cand_id, login_date FROM logins),
days AS (SELECT DISTINCT login_date AS day FROM logins)
SELECT days.day,
       COUNT(DISTINCT d.cand_id) AS users_last_3_days
FROM days
JOIN d ON d.login_date BETWEEN date(days.day, '-2 days') AND days.day
GROUP BY days.day
ORDER BY days.day;
""")

,day,users_last_3_days
0,2026-03-01,3
1,2026-03-02,4
2,2026-03-03,5
3,2026-03-04,5
4,2026-03-05,5
5,2026-03-06,4


### `L4_overqualified`: Non-equi data check

In [93]:
run("""
SELECT ca.name, ca.ctc_lpa, j.job_id, j.max_ctc_lpa
FROM applications a
JOIN candidates ca ON ca.cand_id = a.cand_id
JOIN jobs j        ON j.job_id   = a.job_id
WHERE ca.ctc_lpa > j.max_ctc_lpa
ORDER BY ca.name;
""")

,name,ctc_lpa,job_id,max_ctc_lpa
0,Ananya,15,203,12
1,Diya,18,203,12
2,Rohan,12,205,8
3,Vikram,42,206,40


### `L4_explain_noindex`: Query plan without index

In [94]:
run("""
EXPLAIN QUERY PLAN
SELECT * FROM applications WHERE cand_id = 105;
""")

,id,parent,notused,detail
0,2,0,0,SCAN applications


### `L4_explain_index`: Query plan with index

In [95]:
run("""
CREATE INDEX idx_app_cand ON applications(cand_id);
EXPLAIN QUERY PLAN
SELECT * FROM applications WHERE cand_id = 105;
""")

,id,parent,notused,detail
0,3,0,0,SEARCH applications USING INDEX idx_app_cand (...


### `L4_explain_nonsarg`: Non-sargable filter

In [96]:
run("""
CREATE INDEX idx_app_date ON applications(applied_date);
EXPLAIN QUERY PLAN
SELECT * FROM applications WHERE strftime('%Y', applied_date) = '2026';
""")

,id,parent,notused,detail
0,2,0,0,SCAN applications


### `L4_explain_sarg`: Sargable range

In [97]:
run("""
CREATE INDEX idx_app_date ON applications(applied_date);
EXPLAIN QUERY PLAN
SELECT * FROM applications WHERE applied_date >= '2026-01-01' AND applied_date < '2027-01-01';
""")

,id,parent,notused,detail
0,3,0,0,SEARCH applications USING INDEX idx_app_date (...


### `L4_feature_table`: ML feature table (correct)

In [98]:
run("""
WITH apps AS (                      -- one row per candidate from applications
  SELECT cand_id,
         COUNT(DISTINCT job_id)                         AS n_jobs_applied,
         SUM(CASE WHEN status = 'rejected' THEN 1 ELSE 0 END) AS n_rejected,
         MAX(CASE WHEN status = 'offer'    THEN 1 ELSE 0 END) AS got_offer
  FROM (SELECT DISTINCT cand_id, job_id, applied_date, status FROM applications) a
  GROUP BY cand_id
),
act AS (                            -- one row per candidate from logins
  SELECT cand_id, COUNT(DISTINCT login_date) AS n_login_days
  FROM logins
  GROUP BY cand_id
)
SELECT ca.cand_id,
       ca.exp_yrs,
       COALESCE(ca.ctc_lpa, 0)                 AS ctc_lpa,
       CASE WHEN ca.ctc_lpa IS NULL THEN 1 ELSE 0 END AS ctc_missing,
       COALESCE(ap.n_jobs_applied, 0)          AS n_jobs_applied,
       COALESCE(ap.n_rejected, 0)              AS n_rejected,
       COALESCE(ac.n_login_days, 0)            AS n_login_days,
       COALESCE(ap.got_offer, 0)               AS got_offer
FROM candidates ca
LEFT JOIN apps ap ON ap.cand_id = ca.cand_id
LEFT JOIN act  ac ON ac.cand_id = ca.cand_id
ORDER BY ca.cand_id;
""")

,cand_id,exp_yrs,ctc_lpa,ctc_missing,n_jobs_applied,n_rejected,n_login_days,got_offer
0,101,2,8.0,0,3,0,5,1
1,102,5,18.0,0,3,1,2,1
2,103,0,0.0,1,3,1,0,0
3,104,7,30.0,0,3,0,4,1
4,105,3,12.0,0,3,1,1,0
5,106,1,5.5,0,0,0,0,0
6,107,10,42.0,0,2,1,3,1
7,108,4,15.0,0,2,0,0,0
8,109,2,9.5,0,3,1,2,0
9,110,6,24.0,0,0,0,0,0


### `L4_feature_fanout`: ML feature table (fan-out bug)

In [99]:
run("""
SELECT ca.cand_id,
       SUM(CASE WHEN a.status = 'rejected' THEN 1 ELSE 0 END) AS n_rejected_wrong,
       COUNT(l.login_date)                                   AS n_logins_wrong
FROM candidates ca
LEFT JOIN applications a ON a.cand_id = ca.cand_id
LEFT JOIN logins l       ON l.cand_id = ca.cand_id
WHERE ca.cand_id IN (101, 107, 109)
GROUP BY ca.cand_id
ORDER BY ca.cand_id;
""")

,cand_id,n_rejected_wrong,n_logins_wrong
0,101,0,20
1,107,3,6
2,109,3,9


# Queries used in the interview answers

### `Q_join_counts`: Join row counts puzzle

In [100]:
run("""
WITH a(x) AS (VALUES (1), (1), (2), (NULL)),
     b(x) AS (VALUES (1), (1), (3), (NULL))
SELECT 'inner' AS join_type, COUNT(*) AS n_rows FROM a INNER JOIN b ON a.x = b.x
UNION ALL
SELECT 'left',  COUNT(*) FROM a LEFT  JOIN b ON a.x = b.x
UNION ALL
SELECT 'right', COUNT(*) FROM a RIGHT JOIN b ON a.x = b.x
UNION ALL
SELECT 'full',  COUNT(*) FROM a FULL  JOIN b ON a.x = b.x
UNION ALL
SELECT 'cross', COUNT(*) FROM a CROSS JOIN b;
""")

,join_type,n_rows
0,inner,4
1,left,6
2,right,6
3,full,8
4,cross,16


### `Q_avg_of_avg`: Average of averages

In [101]:
run("""
SELECT (SELECT AVG(salary) FROM employees) AS overall_avg,
       (SELECT AVG(dept_avg)
        FROM (SELECT AVG(salary) AS dept_avg FROM employees GROUP BY dept)) AS avg_of_dept_avgs;
""")

,overall_avg,avg_of_dept_avgs
0,53.0,58.125


### `Q_point_in_time`: Point-in-time features and label

In [102]:
run("""
WITH params AS (SELECT '2026-02-15' AS cutoff),
feat AS (                -- features: only rows strictly before the cutoff
  SELECT a.cand_id,
         COUNT(*) AS apps_before,
         SUM(CASE WHEN a.status = 'rejected' THEN 1 ELSE 0 END) AS rejections_before
  FROM applications a, params p
  WHERE a.applied_date < p.cutoff
  GROUP BY a.cand_id
),
label AS (               -- label: what happened on or after the cutoff
  SELECT a.cand_id,
         MAX(CASE WHEN a.status = 'offer' THEN 1 ELSE 0 END) AS offer_after
  FROM applications a, params p
  WHERE a.applied_date >= p.cutoff
  GROUP BY a.cand_id
)
SELECT c.cand_id,
       COALESCE(f.apps_before, 0)       AS apps_before,
       COALESCE(f.rejections_before, 0) AS rejections_before,
       COALESCE(l.offer_after, 0)       AS label_offer_after
FROM candidates c
LEFT JOIN feat  f ON f.cand_id = c.cand_id
LEFT JOIN label l ON l.cand_id = c.cand_id
WHERE c.signup_date < (SELECT cutoff FROM params)
ORDER BY c.cand_id;
""")

,cand_id,apps_before,rejections_before,label_offer_after
0,101,2,0,1
1,102,3,1,0
2,103,2,1,0
3,104,2,0,0
4,105,3,1,0
5,106,0,0,0


### `Q_lag_selfjoin`: LAG without window functions

In [103]:
run("""
SELECT cur.cand_id, cur.login_date,
       MAX(prev.login_date) AS prev_login
FROM (SELECT DISTINCT cand_id, login_date FROM logins) cur
LEFT JOIN (SELECT DISTINCT cand_id, login_date FROM logins) prev
       ON prev.cand_id = cur.cand_id AND prev.login_date < cur.login_date
WHERE cur.cand_id = 101
GROUP BY cur.cand_id, cur.login_date
ORDER BY cur.login_date;
""")

,cand_id,login_date,prev_login
0,101,2026-03-01,NaN
1,101,2026-03-02,2026-03-01
2,101,2026-03-03,2026-03-02
3,101,2026-03-05,2026-03-03
4,101,2026-03-06,2026-03-05


### `Q_window_in_where`: Filtering on a window function

In [104]:
run("""
SELECT name, salary FROM (
  SELECT name, salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS dr
  FROM employees
) t
WHERE dr <= 3
ORDER BY salary DESC;
""")

,name,salary
0,Neha,90
1,Yash,75
2,Rahul,70


### `Q_city_share`: Share by city

In [105]:
run("""
SELECT COALESCE(city, 'Unknown') AS city,
       COUNT(*) AS n_cands,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM candidates), 1) AS pct
FROM candidates
GROUP BY COALESCE(city, 'Unknown')
ORDER BY n_cands DESC, city;
""")

,city,n_cands,pct
0,Bengaluru,3,30.0
1,Delhi,2,20.0
2,Mumbai,2,20.0
3,Noida,1,10.0
4,Pune,1,10.0
5,Unknown,1,10.0


### `Q_applied_outside_city`: Applied outside home city

In [106]:
run("""
SELECT ca.name, ca.city AS home_city, j.city AS job_city, j.title
FROM applications a
JOIN candidates ca ON ca.cand_id = a.cand_id
JOIN jobs j        ON j.job_id   = a.job_id
WHERE ca.city <> j.city
  AND a.app_id <> 22
ORDER BY ca.name, j.job_id;
""")

,name,home_city,job_city,title
0,Aarav,Bengaluru,Noida,Data Scientist
1,Diya,Mumbai,Bengaluru,Data Scientist
2,Ishaan,Bengaluru,Noida,Data Scientist
3,Kabir,Delhi,Mumbai,Data Analyst
4,Kabir,Delhi,Pune,Data Analyst
5,Kabir,Delhi,Bengaluru,Data Analyst
6,Meera,Bengaluru,Noida,ML Engineer
7,Rohan,Pune,Bengaluru,Data Scientist
8,Rohan,Pune,Mumbai,Data Scientist
9,Vikram,Noida,Bengaluru,Data Scientist


### `Q_company_offer_rate`: Offer rate per company

In [107]:
run("""
SELECT co.name AS company,
       COUNT(a.app_id) AS applications,
       SUM(CASE WHEN a.status = 'offer' THEN 1 ELSE 0 END) AS offers,
       ROUND(1.0 * SUM(CASE WHEN a.status = 'offer' THEN 1 ELSE 0 END)
             / NULLIF(COUNT(a.app_id), 0), 2) AS offer_rate
FROM companies co
LEFT JOIN jobs j          ON j.company_id = co.company_id
LEFT JOIN applications a  ON a.job_id     = j.job_id AND a.app_id <> 22
GROUP BY co.company_id, co.name
ORDER BY offer_rate DESC, company;
""")

,company,applications,offers,offer_rate
0,EduSpark,4,1,0.25
1,DataNest,5,1,0.20
2,ShopKart,5,1,0.20
3,PayWave,6,1,0.17
4,MediCore,2,0,0.00
5,GreenGrid,0,0,NaN


# Practice exercises
Write your query in each cell (replace the `...`), run it, then compare with the solutions section.

**Exercise 1.** Which companies are in Bengaluru? Show name and industry.

In [108]:
# run("""
# ... your query ...
# """)

**Exercise 2.** List candidates with 3 to 6 years of experience, highest CTC first. Put candidates with unknown CTC last.

In [109]:
# run("""
# ... your query ...
# """)

**Exercise 3.** How many candidates did not fill in their city? (Answer: 1)

In [110]:
# run("""
# ... your query ...
# """)

**Exercise 4.** For each job title, how many applications were received? Include titles with zero.

In [111]:
# run("""
# ... your query ...
# """)

**Exercise 5.** Which candidates have applied to at least two different companies?

In [112]:
# run("""
# ... your query ...
# """)

**Exercise 6.** Rejection rate per candidate (rejections / applications), excluding the duplicate row 22.

In [113]:
# run("""
# ... your query ...
# """)

**Exercise 7.** Which jobs received no applications? Write it with NOT EXISTS.

In [114]:
# run("""
# ... your query ...
# """)

**Exercise 8.** For each candidate who was referred, show the referrer's name and whether the referrer has an offer.

In [115]:
# run("""
# ... your query ...
# """)

**Exercise 9.** Highest-paid employee in each department (all of them if tied).

In [116]:
# run("""
# ... your query ...
# """)

**Exercise 10.** Each candidate's most recent application (one row per candidate).

In [117]:
# run("""
# ... your query ...
# """)

**Exercise 11.** Number of days between each candidate's signup and their first application.

In [118]:
# run("""
# ... your query ...
# """)

**Exercise 12.** Each employee's salary as a percentage of their department's total.

In [119]:
# run("""
# ... your query ...
# """)

**Exercise 13.** Longest login streak of every candidate who logged in at all.

In [120]:
# run("""
# ... your query ...
# """)

**Exercise 14.** Applications per company per month as a pivot (months as columns).

In [121]:
# run("""
# ... your query ...
# """)

**Exercise 15.** Click-through rate per candidate in the A/B test, and the variant each candidate was in. Was assignment per candidate or per impression?

In [122]:
# run("""
# ... your query ...
# """)

# Solutions

**Solution 1.** Which companies are in Bengaluru? Show name and industry.

In [123]:
run("""
SELECT name, industry FROM companies WHERE city = 'Bengaluru';
""")

,name,industry
0,DataNest,IT
1,ShopKart,E-commerce


**Solution 2.** List candidates with 3 to 6 years of experience, highest CTC first. Put candidates with unknown CTC last.

In [124]:
run("""
SELECT name, exp_yrs, ctc_lpa FROM candidates
WHERE exp_yrs BETWEEN 3 AND 6
ORDER BY ctc_lpa IS NULL, ctc_lpa DESC;
""")

,name,exp_yrs,ctc_lpa
0,Priya,6,24
1,Diya,5,18
2,Ananya,4,15
3,Rohan,3,12


**Solution 3.** How many candidates did not fill in their city? (Answer: 1)

In [125]:
run("""
SELECT COUNT(*) - COUNT(city) AS missing_city FROM candidates;
""")

,missing_city
0,1


**Solution 4.** For each job title, how many applications were received? Include titles with zero.

In [126]:
run("""
SELECT j.title, COUNT(a.app_id) AS n_apps
FROM jobs j LEFT JOIN applications a ON a.job_id = j.job_id
GROUP BY j.title ORDER BY n_apps DESC;
""")

,title,n_apps
0,Data Scientist,12
1,Data Analyst,8
2,ML Engineer,3


**Solution 5.** Which candidates have applied to at least two different companies?

In [127]:
run("""
SELECT a.cand_id, COUNT(DISTINCT j.company_id) AS n_companies
FROM applications a JOIN jobs j ON j.job_id = a.job_id
GROUP BY a.cand_id HAVING COUNT(DISTINCT j.company_id) >= 2;
""")

,cand_id,n_companies
0,101,3
1,102,2
2,103,3
3,104,3
4,105,3
5,107,2
6,109,3


**Solution 6.** Rejection rate per candidate (rejections / applications), excluding the duplicate row 22.

In [128]:
run("""
SELECT cand_id, COUNT(*) AS apps,
       ROUND(AVG(CASE WHEN status = 'rejected' THEN 1.0 ELSE 0 END), 2) AS rejection_rate
FROM applications WHERE app_id <> 22
GROUP BY cand_id ORDER BY rejection_rate DESC, cand_id;
""")

,cand_id,apps,rejection_rate
0,107,2,0.50
1,102,3,0.33
2,103,3,0.33
3,105,3,0.33
4,109,3,0.33
5,101,3,0.00
6,104,3,0.00
7,108,2,0.00


**Solution 7.** Which jobs received no applications? Write it with NOT EXISTS.

In [129]:
run("""
SELECT job_id, title FROM jobs j
WHERE NOT EXISTS (SELECT 1 FROM applications a WHERE a.job_id = j.job_id);
""")

,job_id,title
0,210,ML Engineer


**Solution 8.** For each candidate who was referred, show the referrer's name and whether the referrer has an offer.

In [130]:
run("""
SELECT c.name AS candidate, r.name AS referrer,
       CASE WHEN EXISTS (SELECT 1 FROM applications a WHERE a.cand_id = r.cand_id AND a.status = 'offer')
            THEN 'yes' ELSE 'no' END AS referrer_has_offer
FROM candidates c JOIN candidates r ON r.cand_id = c.referred_by;
""")

,candidate,referrer,referrer_has_offer
0,Diya,Aarav,yes
1,Meera,Diya,yes
2,Rohan,Aarav,yes
3,Vikram,Meera,yes
4,Ishaan,Rohan,no


**Solution 9.** Highest-paid employee in each department (all of them if tied).

In [131]:
run("""
SELECT dept, name, salary FROM (
  SELECT *, RANK() OVER (PARTITION BY dept ORDER BY salary DESC) AS r FROM employees WHERE salary IS NOT NULL
) t WHERE r = 1 ORDER BY dept;
""")

,dept,name,salary
0,DataScience,Arjun,60
1,Engineering,Yash,75
2,Leadership,Neha,90
3,Sales,Pooja,40


**Solution 10.** Each candidate's most recent application (one row per candidate).

In [132]:
run("""
SELECT cand_id, job_id, applied_date, status FROM (
  SELECT a.*, ROW_NUMBER() OVER (PARTITION BY cand_id ORDER BY applied_date DESC, app_id DESC) AS rn
  FROM applications a
) t WHERE rn = 1 ORDER BY cand_id;
""")

,cand_id,job_id,applied_date,status
0,101,209,2026-03-05,applied
1,102,204,2026-02-06,offer
2,103,207,2026-02-19,applied
3,104,208,2026-03-02,applied
4,105,205,2026-02-11,rejected
5,107,208,2026-03-02,offer
6,108,204,2026-02-22,shortlisted
7,109,209,2026-03-05,shortlisted


**Solution 11.** Number of days between each candidate's signup and their first application.

In [133]:
run("""
SELECT c.cand_id, c.signup_date, MIN(a.applied_date) AS first_app,
       CAST(julianday(MIN(a.applied_date)) - julianday(c.signup_date) AS INTEGER) AS days_to_first_app
FROM candidates c JOIN applications a ON a.cand_id = c.cand_id
GROUP BY c.cand_id, c.signup_date ORDER BY days_to_first_app;
""")

,cand_id,signup_date,first_app,days_to_first_app
0,105,2026-02-01,2026-02-02,1
1,107,2026-02-15,2026-02-16,1
2,108,2026-02-20,2026-02-21,1
3,109,2026-03-02,2026-03-03,1
4,104,2026-01-12,2026-01-21,9
5,101,2026-01-05,2026-01-16,11
6,102,2026-01-07,2026-01-18,11
7,103,2026-01-10,2026-01-27,17


**Solution 12.** Each employee's salary as a percentage of their department's total.

In [134]:
run("""
SELECT name, dept, salary,
       ROUND(100.0 * salary / SUM(salary) OVER (PARTITION BY dept), 1) AS pct_of_dept
FROM employees WHERE salary IS NOT NULL ORDER BY dept, pct_of_dept DESC;
""")

,name,dept,salary,pct_of_dept
0,Arjun,DataScience,60,33.3
1,Kavya,DataScience,45,25.0
2,Manav,DataScience,45,25.0
3,Tara,DataScience,30,16.7
4,Yash,Engineering,75,38.5
5,Rahul,Engineering,70,35.9
6,Sneha,Engineering,50,25.6
7,Neha,Leadership,90,100.0
8,Pooja,Sales,40,61.5
9,Dev,Sales,25,38.5


**Solution 13.** Longest login streak of every candidate who logged in at all.

In [135]:
run("""
WITH d AS (SELECT DISTINCT cand_id, login_date FROM logins),
g AS (SELECT cand_id, julianday(login_date) - ROW_NUMBER() OVER (PARTITION BY cand_id ORDER BY login_date) AS grp FROM d),
s AS (SELECT cand_id, grp, COUNT(*) AS len FROM g GROUP BY cand_id, grp)
SELECT cand_id, MAX(len) AS longest_streak FROM s GROUP BY cand_id ORDER BY longest_streak DESC, cand_id;
""")

,cand_id,longest_streak
0,104,4
1,101,3
2,107,3
3,109,2
4,102,1
5,105,1


**Solution 14.** Applications per company per month as a pivot (months as columns).

In [136]:
run("""
SELECT co.name,
  SUM(CASE WHEN strftime('%m', a.applied_date) = '01' THEN 1 ELSE 0 END) AS jan,
  SUM(CASE WHEN strftime('%m', a.applied_date) = '02' THEN 1 ELSE 0 END) AS feb,
  SUM(CASE WHEN strftime('%m', a.applied_date) = '03' THEN 1 ELSE 0 END) AS mar
FROM companies co
LEFT JOIN jobs j ON j.company_id = co.company_id
LEFT JOIN applications a ON a.job_id = j.job_id AND a.app_id <> 22
GROUP BY co.company_id, co.name;
""")

,name,jan,feb,mar
0,DataNest,3,1,1
1,PayWave,2,4,0
2,MediCore,0,2,0
3,ShopKart,0,4,1
4,EduSpark,0,0,4
5,GreenGrid,0,0,0


**Solution 15.** Click-through rate per candidate in the A/B test, and the variant each candidate was in. Was assignment per candidate or per impression?

In [137]:
run("""
SELECT cand_id, COUNT(DISTINCT variant) AS n_variants, MIN(variant) AS variant,
       ROUND(AVG(clicked * 1.0), 2) AS ctr
FROM impressions GROUP BY cand_id ORDER BY cand_id;  -- n_variants = 1 for all: per candidate
""")

,cand_id,n_variants,variant,ctr
0,101,1,A,0.5
1,102,1,B,0.5
2,103,1,A,0.0
3,104,1,B,1.0
4,105,1,A,0.5
5,107,1,B,0.5
6,108,1,A,0.0
7,109,1,B,0.5
